# R5.5 - Classifier Collapse Diagnostic

Investigates WHY the trained classifier collapses to coconut (val acc 0.5584 /
macro-F1 0.1791 = the trivial majority-class anchor) before resuming any
hyperparameter sweep. Uses the frozen R5.2.9-enriched corpus
`crop_supervised_v2.csv` + manifest `crop_supervised_v2.0_manifest.json`
(train 5924 / val 2459 / test 2291).

Runs the GPU-backed diagnostic phases:

- Phases 5/9  image separability + model-input normalization stats
- Phase 3     binary coconut-vs-pepper training (3 loss variants)
- Phase 10    first-N-step training dynamics (softmax collapse probe)
- Phase 11    tiny-set (20+20) overfit capacity probe
- Phases 12-14 sklearn baselines (tabular / imagery / combined)

Tabular-only phases (1,2,4,6,7,8,16,17) ran locally via
`diagnose_collapse.py`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Brijesh2005/CropPrep.git'
REPO_ROOT = Path('/kaggle/working/CropPrep')

if not (REPO_ROOT / '.git').exists():
    print(f'cloning CropPrep -> {REPO_ROOT}')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'repo root: {REPO_ROOT}')
print(f'cwd: {os.getcwd()}')

## 0.1 P100 GPU fix

Reinstall torch from cu126 index for Pascal P100 compatibility.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'],
    check=False,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
     '--index-url', 'https://download.pytorch.org/whl/cu126'],
    check=True,
)
import torch
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')

## 1. Environment verification

Verify GPU availability. STOP if unavailable.

In [ ]:
import torch, sys

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
else:
    raise RuntimeError('GPU REQUIRED FOR R5.5 DIAGNOSTIC')

## 2. Frozen Data Contract Gate

Verify the frozen R5.2.9-enriched corpus before any build.

In [ ]:
import json, csv

manifest_path = REPO_ROOT / 'training_manifests' / 'crop_supervised_v2.0_manifest.json'
csv_path = REPO_ROOT / 'govt_crop_matched_v2' / 'crop_supervised_v2.csv'

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
declared = list(manifest.get('supervised_classes') or [])
print('supervised_classes:', declared)
print('contract:', {k: manifest[k] for k in ('total_samples', 'train_samples', 'validation_samples', 'test_samples')})

with open(csv_path, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

eligible = [r for r in rows if str(r.get('benchmark_eligible', '')).lower() in ('1', 'true', 'yes')]
from collections import Counter
print('benchmark-eligible rows:', len(eligible))
print('eligible crop counts:', dict(Counter(r['crop'].strip().lower() for r in eligible)))

## 2b. Imagery window (P100 CPU-safe defaults)

Same acquisition window as the R5.3 campaign parking lot so the diagnostic
sees the exact model input: `ST_IMAGERY__MODE=window_days`, 180 days,
closest-to-survey, up to 8 observations.

In [ ]:
import os
os.environ.setdefault('ST_IMAGERY__MODE', 'window_days')
os.environ.setdefault('ST_IMAGERY__WINDOW_DAYS', '180')
os.environ.setdefault('ST_IMAGERY__START_MONTH', '5')
os.environ.setdefault('ST_IMAGERY__SPAN_MONTHS', '12')
os.environ.setdefault('ST_IMAGERY__STRATEGY', 'closest_to_survey')
os.environ.setdefault('ST_IMAGERY__MAX_OBSERVATIONS', '8')
print('Imagery window:', os.environ['ST_IMAGERY__MODE'],
      'days=' + os.environ['ST_IMAGERY__WINDOW_DAYS'],
      'strategy=' + os.environ['ST_IMAGERY__STRATEGY'])

## 3. Bootstrap + System Check

In [ ]:
!python training/kaggle/scripts/bootstrap.py --skip-install
!python training/kaggle/scripts/system_check.py
!python training/kaggle/scripts/gpu_smoke_test.py

## 4. Classifier-Collapse Diagnostic

In [ ]:
!python training/kaggle/scripts/diagnose_collapse_kaggle.py \
    --output training/kaggle/outputs/reports; echo "diagnose_exit_status=$?"

## 5. Diagnostic summary

In [ ]:
import json
from pathlib import Path

rep_path = REPO_ROOT / 'training/kaggle/outputs/reports/diagnostic_r5_5.json'
if not rep_path.exists():
    raise RuntimeError('diagnostic_r5_5.json missing')

d = json.loads(rep_path.read_text())
corpus = d.get('corpus', {})
print('=' * 60)
print('  R5.5 CLASSIFIER-COLLAPSE DIAGNOSTIC')
print('=' * 60)
print(f"  corpus: total={corpus.get('total')} train={corpus.get('train')} "
      f"val={corpus.get('val')} test={corpus.get('test')}")
class_counts = corpus.get('class_counts', {})
if class_counts.get('train'):
    print('  train class counts:', class_counts['train'])
if class_counts.get('val'):
    print('  val class counts:  ', class_counts['val'])
print('  class weights:', json.dumps(d.get('class_weights', {}), default=str))

binary = d.get('binary', {})
if binary.get('variants'):
    print('\n -- binary coconut-vs-pepper --')
    print(f"  val majority prior accuracy: {binary.get('binary_val_majority_acc')}")
    for v in binary['variants']:
        m = v.get('metrics', {})
        print(f"  {v['variant']}: acc={m.get('accuracy')} macro_f1={m.get('macro_f1')} "
              f"beats_prior={m.get('beats_majority_prior')}")
        print(f"    pred_dist={m.get('prediction_distribution')}")

dyn = d.get('dynamics', {})
if dyn.get('steps'):
    print('\n -- Phase 10 dynamics (first steps) --')
    for s in dyn['steps'][:5]:
        print(f"  step {s['step']}: loss={s.get('train_loss')} "
              f"softmax={s.get('mean_softmax')}")

tiny = d.get('tiny_overfit', {})
if tiny:
    print('\n -- Phase 11 tiny overfit --')
    print('  steps_to_acc_090:', tiny.get('steps_to_acc_090'))

print('\n  wrote:', rep_path)

## 6. Status

In [ ]:
print('R5.5 diagnostic notebook completed')
print('  report: training/kaggle/outputs/reports/diagnostic_r5_5.json')